## load_zillow_long
Transposes the three Zillow **wide** CSVs (5 id columns + one date column per month) into **long-form Parquet** datasets, one per feed, under `RAW_ZILLOW_LONG` (`/Volumes/{CATALOG}/raw/zillow/_long/`). Runs after `download_sources` in the `download_sources` job; the future Bronze loader reads these `_long` datasets.

Design: `_dev_planning/design_docs/zillow_wide_to_long_step_design.md`. Write strategy: **overwrite** (full-snapshot, idempotent). No archiving (rolling source files).

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injected: RAW_ZILLOW, RAW_ZILLOW_LONG, ZILLOW_FEEDS, StepLog, AUDIT,
# PIPELINE_RUN_ID, Utils, spark, dbutils, F. wide_to_long resolves via shared_lib_path.
from wide_to_long.api import read_and_unpivot

STEP_SEQUENCE = 2
# Zillow wide layout: 5 leading id columns (RegionID, SizeRank, RegionName, RegionType,
# StateName); every remaining column is a YYYY-MM-DD monthly date. Used only for the
# row-count assertion baseline (expected long rows = wide_rows x date_cols).
N_ID_COLS = 5

In [ ]:
# Open the pipeline_step_log row (RUNNING). target_table=None: this step writes _long
# Parquet datasets to a Volume, not a managed table.
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "bronze",
    target_table    = None,
)
print(f"load_zillow_long: step_log_id={step.step_log_id}")

In [ ]:
# Transpose each feed wide->long and overwrite its _long Parquet dataset. One StepLog
# row covers the whole step; a missing/bad file fails loudly (download is an
# ALL_SUCCESS upstream dependency, so the files must be present).
try:
    total_read = 0
    total_written = 0
    for feed, stem in ZILLOW_FEEDS.items():
        wide_path = f"{RAW_ZILLOW}{stem}.csv"
        long_path = f"{RAW_ZILLOW_LONG}{stem}_long"

        # Baseline shape for the row-count assertion (all-string read, no inference).
        wide = spark.read.option("header", True).option("inferSchema", False).csv(wide_path)
        n_rows  = wide.count()
        n_dates = len(wide.columns) - N_ID_COLS

        long_df  = read_and_unpivot(spark, wide_path)
        n_long   = long_df.count()
        expected = n_rows * n_dates
        if n_long != expected:
            raise AssertionError(
                f"[{feed}] wide->long row-count mismatch: produced {n_long:,}, expected "
                f"{n_rows:,} rows x {n_dates} date cols = {expected:,} ({wide_path})."
            )

        long_df.write.mode("overwrite").parquet(long_path)
        total_read    += n_rows
        total_written += n_long
        print(f"load_zillow_long: {feed}: {n_rows:,} x {n_dates} -> {n_long:,} long rows -> {long_path}")

    step.rows_read    = total_read
    step.rows_written = total_written
    step.succeed()
    print(f"load_zillow_long: DONE read={total_read:,} written={total_written:,}")
except Exception as e:
    step.fail(e); raise